# Desafio Cientista de Dados — Carpa Analytics (Jul/2026)

**Base:** empréstimos intermediados pela LendingClub em um determinado período.

**Alvo:** `is_bad` (1 = inadimplente, 0 = bom pagador).

Este notebook responde às duas primeiras perguntas do desafio:

1. Viés de amostragem nos dados.
2. Qualidade do `grade` da LendingClub como indicador de risco de inadimplência.

Todas as análises foram implementadas em funções documentadas (docstrings no padrão PEP), para
que o notebook funcione como uma pequena biblioteca de análise reutilizável, e não como uma
sequência de comandos soltos.


## 1. Configuração e imports

In [ ]:
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11


In [ ]:
@dataclass
class ConfiguracaoAnalise:
    """Configuração central da análise.

    Parameters
    ----------
    caminho_dados : Path
        Caminho para o arquivo CSV com os dados de treino.
    caminho_saida_pdf : Path
        Caminho onde o PDF consolidado de gráficos será salvo.
    coluna_alvo : str
        Nome da coluna que indica inadimplência (variável resposta).
    ordem_grades : list[str]
        Ordem crescente de risco das categorias de `grade`, da melhor (A)
        para a pior (G). Usada em todos os gráficos e testes que dependem
        da ordinalidade do grade.
    cor_bom : str
        Cor usada para representar bons pagadores nos gráficos.
    cor_mau : str
        Cor usada para representar maus pagadores (inadimplentes) nos gráficos.
    """

    caminho_dados: Path = Path("data/LendingClub_Isbad_Treino.csv")
    caminho_saida_pdf: Path = Path("graficos_analise.pdf")
    coluna_alvo: str = "is_bad"
    ordem_grades: list = field(default_factory=lambda: ["A", "B", "C", "D", "E", "F", "G"])
    cor_bom: str = "#2E7D32"
    cor_mau: str = "#C62828"


CFG = ConfiguracaoAnalise()
FIGURAS_PARA_PDF = []


## 2. Carga e tratamento dos dados

O CSV traz alguns campos que precisam de tratamento antes de qualquer análise: `term` vem como
string com espaço (`" 60 months"`), `int_rate` vem como string, e várias colunas de meses
(`mths_since_*`) usam ausência de valor para representar "nunca aconteceu", o que é diferente de
dado faltante por erro de coleta.


In [ ]:
def carregar_dados(caminho: Path) -> pd.DataFrame:
    """Carrega o CSV bruto de empréstimos da LendingClub.

    Parameters
    ----------
    caminho : Path
        Caminho para o arquivo CSV de treino.

    Returns
    -------
    pandas.DataFrame
        DataFrame com os dados exatamente como estão no arquivo, sem
        nenhum tratamento de tipo. O tratamento é responsabilidade da
        função `tratar_tipos`, para manter as duas etapas independentes
        e testáveis separadamente.

    Raises
    ------
    FileNotFoundError
        Se o arquivo não existir no caminho informado.
    """
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo de dados não encontrado em '{caminho}'. "
            "Confira se o CSV está na pasta 'data/'."
        )
    return pd.read_csv(caminho, low_memory=False)


def tratar_tipos(df_bruto: pd.DataFrame) -> pd.DataFrame:
    """Aplica conversões de tipo e limpeza leve nas colunas usadas na análise.

    Parameters
    ----------
    df_bruto : pandas.DataFrame
        DataFrame retornado por `carregar_dados`, sem tratamento.

    Returns
    -------
    pandas.DataFrame
        Cópia do DataFrame de entrada com as seguintes colunas tratadas:
        `term` (int, em meses), `int_rate` (float, em percentual),
        `issue_d` e `earliest_cr_line` (datetime), `grade` (categórica
        ordenada de A a G) e `emp_length` (float, anos de emprego, com
        "10+ years" mapeado para 10 e "< 1 year" para 0).

    Notes
    -----
    A ordinalidade do `grade` é fixada aqui via `pd.Categorical` com
    `ordered=True`, o que permite comparações diretas (`df.grade < "C"`)
    e é aproveitado depois nas funções de correlação e IV.
    """
    df = df_bruto.copy()

    df["term"] = df["term"].astype(str).str.extract(r"(\d+)").astype(int)
    df["int_rate"] = pd.to_numeric(df["int_rate"], errors="coerce")

    df["issue_d"] = pd.to_datetime(df["issue_d"], errors="coerce")
    df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], errors="coerce")

    df["grade"] = pd.Categorical(df["grade"], categories=CFG.ordem_grades, ordered=True)

    mapa_emp_length = {
        "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
        "10+ years": 10,
    }
    df["emp_length_anos"] = df["emp_length"].map(mapa_emp_length)

    df[CFG.coluna_alvo] = df[CFG.coluna_alvo].astype(int)

    return df


df_bruto = carregar_dados(CFG.caminho_dados)
df = tratar_tipos(df_bruto)
print(f"Shape: {df.shape}")
df.head(3)


In [ ]:
def resumo_geral(df: pd.DataFrame) -> pd.DataFrame:
    """Gera um resumo tabular de tipos e completude das colunas.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame já tratado.

    Returns
    -------
    pandas.DataFrame
        Uma linha por coluna, com tipo, quantidade de nulos, percentual
        de nulos e número de valores únicos. Ordenado pelo percentual de
        nulos em ordem decrescente, para destacar rapidamente as colunas
        mais problemáticas.
    """
    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(1),
        "n_unicos": df.nunique(),
    })
    return resumo.sort_values("pct_nulos", ascending=False)


resumo_geral(df).head(15)


## 3. Pergunta 1 — Existe viés de amostragem?

O enunciado é claro ao dizer que os dados são referentes aos empréstimos **intermediados** pela
LendingClub. Isso já é, por si só, um forte indício de viés de seleção: a base não representa "o
universo de pessoas que precisam de crédito", e sim um subconjunto bem específico, filtrado em
várias etapas:

1. **Viés de auto-seleção**: só está na base quem decidiu solicitar um empréstimo via LendingClub,
   uma plataforma de crédito P2P online. Isso já exclui quem não tem acesso à internet, quem não
   confia em plataformas online de crédito, ou quem prefere canais bancários tradicionais.
2. **Viés de aprovação (survivorship bias)**: a coluna `is_bad` só faz sentido para empréstimos que
   foram de fato **fundidos** (funded). Pedidos rejeitados pela LendingClub (por score de crédito
   baixo, renda não comprovada, etc.) simplesmente não aparecem na base. Ou seja, estamos olhando
   só para quem já passou por um filtro de crédito prévio, o que tende a subestimar o risco real da
   população de solicitantes.
3. **Viés geográfico e regulatório**: por razões regulatórias, historicamente a LendingClub não
   operava em todos os estados americanos. A distribuição por `addr_state` tende a refletir isso,
   e não a distribuição real de tomadores de crédito no país.
4. **Viés temporal / de janela de observação (censura)**: como os dados cobrem "um determinado
   período de tempo", é bem provável que existam empréstimos emitidos perto do fim da janela de
   coleta que ainda não tiveram tempo de "maturar" (ex.: um empréstimo de 60 meses emitido há 3
   meses). Se `is_bad` for calculado sobre o estado atual do empréstimo, esses casos recentes
   tendem a aparecer artificialmente como "bom pagador" só porque ainda não deu tempo de
   inadimplir. Isso é um viés de censura, e pode ser testado empiricamente olhando a taxa de
   inadimplência por safra de emissão.
5. **Viés de informação auto-declarada**: campos como `annual_inc`, `emp_title` e `emp_length` são
   auto-reportados pelo tomador e nem sempre verificados (`verification_status`), o que introduz
   ruído sistemático (tendência a superestimar renda, por exemplo).

Abaixo, verificamos empiricamente os pontos 3 e 4 com gráficos, que são os que dá pra checar
diretamente nos dados disponíveis.


In [ ]:
def plot_volume_emissao_temporal(df: pd.DataFrame) -> plt.Figure:
    """Plota o volume de empréstimos emitidos por mês (`issue_d`).

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado, contendo a coluna `issue_d` já convertida
        para datetime.

    Returns
    -------
    matplotlib.figure.Figure
        Figura com a série temporal de volume de emissões, usada para
        evidenciar que a base não é uniformemente distribuída no tempo
        (plataformas de crédito P2P cresceram fortemente ano a ano nesse
        período).
    """
    serie = df.set_index("issue_d").resample("MS").size()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(serie.index, serie.values, color="#1565C0", linewidth=2)
    ax.fill_between(serie.index, serie.values, color="#1565C0", alpha=0.15)
    ax.set_title("Volume de empréstimos emitidos por mês")
    ax.set_xlabel("Mês de emissão (issue_d)")
    ax.set_ylabel("Quantidade de empréstimos")
    fig.tight_layout()
    return fig


fig = plot_volume_emissao_temporal(df)
FIGURAS_PARA_PDF.append(fig)
plt.show()


In [ ]:
def plot_taxa_inadimplencia_por_safra(df: pd.DataFrame, coluna_alvo: str = CFG.coluna_alvo) -> plt.Figure:
    """Plota a taxa de inadimplência observada por safra mensal de emissão.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado, contendo `issue_d` e a coluna alvo.
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.

    Returns
    -------
    matplotlib.figure.Figure
        Gráfico de linha com a taxa de `is_bad` por safra de emissão.
        Uma queda consistente na taxa de inadimplência nas safras mais
        recentes é evidência de viés de censura: empréstimos mais novos
        não tiveram tempo de maturar e de fato inadimplir.

    Notes
    -----
    Safras com poucos contratos (menos de 30) são descartadas do gráfico
    para evitar que ruído estatístico de amostras pequenas seja
    confundido com o efeito de censura que queremos evidenciar.
    """
    agrupado = (
        df.set_index("issue_d")
        .resample("MS")[coluna_alvo]
        .agg(["mean", "count"])
        .rename(columns={"mean": "taxa_inadimplencia", "count": "n_contratos"})
    )
    agrupado = agrupado[agrupado["n_contratos"] >= 30]

    fig, ax1 = plt.subplots(figsize=(11, 4.5))
    ax1.plot(agrupado.index, agrupado["taxa_inadimplencia"] * 100, color=CFG.cor_mau,
             linewidth=2, marker="o", markersize=3, label="Taxa de inadimplência")
    ax1.set_ylabel("Taxa de inadimplência (%)", color=CFG.cor_mau)
    ax1.tick_params(axis="y", labelcolor=CFG.cor_mau)
    ax1.set_xlabel("Mês de emissão (issue_d)")
    ax1.set_title("Taxa de inadimplência observada por safra de emissão")

    ax2 = ax1.twinx()
    ax2.bar(agrupado.index, agrupado["n_contratos"], width=20, color="grey", alpha=0.25,
            label="Nº de contratos")
    ax2.set_ylabel("Nº de contratos na safra", color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")

    fig.tight_layout()
    return fig


fig = plot_taxa_inadimplencia_por_safra(df)
FIGURAS_PARA_PDF.append(fig)
plt.show()


In [ ]:
def plot_distribuicao_categorica(
    df: pd.DataFrame,
    coluna: str,
    titulo: str,
    top_n: int = 15,
    ordem: list = None,
) -> plt.Figure:
    """Plota a distribuição de frequência de uma coluna categórica.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna : str
        Nome da coluna categórica a ser plotada.
    titulo : str
        Título do gráfico.
    top_n : int, opcional
        Número máximo de categorias exibidas, ordenadas por frequência.
        Ignorado se `ordem` for informado.
    ordem : list, opcional
        Ordem explícita das categorias no eixo X (por exemplo, para
        respeitar a ordinalidade do `grade`). Se None, as categorias são
        ordenadas por frequência decrescente.

    Returns
    -------
    matplotlib.figure.Figure
        Gráfico de barras com a distribuição percentual da coluna,
        usado para evidenciar concentrações que sugerem viés amostral
        (por exemplo, concentração geográfica em poucos estados).
    """
    contagem = df[coluna].value_counts(normalize=True, dropna=False) * 100

    if ordem is not None:
        contagem = contagem.reindex(ordem)
    else:
        contagem = contagem.sort_values(ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(contagem.index.astype(str), contagem.values, color="#1565C0")
    ax.set_title(titulo)
    ax.set_ylabel("Percentual da base (%)")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    return fig


fig = plot_distribuicao_categorica(df, "addr_state", "Distribuição geográfica dos empréstimos (top 15 estados)")
FIGURAS_PARA_PDF.append(fig)
plt.show()


**Leitura dos gráficos acima:**

O volume de emissões cresce fortemente ao longo do tempo (típico de uma fintech em expansão), o
que já concentra a base nos períodos mais recentes da janela observada. Combinado a isso, a taxa
de inadimplência por safra tende a cair exatamente nas últimas safras, o padrão esperado de viés de
censura: quanto mais recente o empréstimo, menos tempo ele teve para "dar errado", então ele é
observado como bom pagador mesmo que ainda vá inadimplir depois do corte dos dados. Isso é um
problema real para qualquer modelo treinado direto em cima de `is_bad` sem considerar o tempo de
maturação de cada contrato. A concentração geográfica em poucos estados reforça o viés de
representatividade: a amostra não é um retrato da população americana de tomadores de crédito, é o
retrato de quem tinha acesso à LendingClub nesses estados, nesse período, e que passou pelo crivo
de aprovação da empresa.


## 4. Pergunta 2 — O `grade` é um bom indicador de risco?

A ideia aqui é avaliar o `grade` como se fosse um **score de risco categórico e ordinal** (A é o
melhor, G é o pior) e checar se ele de fato separa bons e maus pagadores. Para isso, combinamos
quatro abordagens complementares:

1. **Gráfica**: taxa de inadimplência por grade, com intervalo de confiança, para ver se a relação
   é monotônica (o grade deveria piorar de forma consistente de A para G).
2. **Teste de associação**: qui-quadrado de independência e Cramér's V, para saber se a associação
   entre `grade` e `is_bad` é estatisticamente significativa e qual o tamanho do efeito.
3. **Correlação ordinal**: Spearman entre o rank do grade e `is_bad`, que captura diretamente se a
   ordem do grade é respeitada pela inadimplência observada.
4. **Poder discriminatório tipo score**: AUC/Gini tratando o grade como um score de risco, e uma
   curva de lift acumulado, no estilo do que se usa para avaliar modelos de crédito.


In [ ]:
def calcular_taxa_inadimplencia_por_categoria(
    df: pd.DataFrame,
    coluna_categoria: str,
    coluna_alvo: str = CFG.coluna_alvo,
    ordem: list = None,
) -> pd.DataFrame:
    """Calcula a taxa de inadimplência por categoria, com intervalo de confiança.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna_categoria : str
        Nome da coluna categórica usada para agrupar (ex.: `grade`).
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.
    ordem : list, opcional
        Ordem em que as categorias devem aparecer no resultado. Se None,
        usa a ordem natural das categorias (relevante para colunas já
        declaradas como `Categorical` ordenado).

    Returns
    -------
    pandas.DataFrame
        Uma linha por categoria, com número de contratos, taxa de
        inadimplência, e limites inferior/superior do intervalo de
        confiança de 95% (aproximação normal para proporção).
    """
    agrupado = df.groupby(coluna_categoria, observed=True)[coluna_alvo].agg(["mean", "count"])
    agrupado = agrupado.rename(columns={"mean": "taxa_inadimplencia", "count": "n_contratos"})

    z = 1.96
    p = agrupado["taxa_inadimplencia"]
    n = agrupado["n_contratos"]
    erro_padrao = np.sqrt(p * (1 - p) / n)
    agrupado["ic_inferior"] = (p - z * erro_padrao).clip(lower=0)
    agrupado["ic_superior"] = (p + z * erro_padrao).clip(upper=1)

    if ordem is not None:
        agrupado = agrupado.reindex(ordem)

    return agrupado.reset_index()


tabela_grade = calcular_taxa_inadimplencia_por_categoria(df, "grade", ordem=CFG.ordem_grades)
tabela_grade


In [ ]:
def plot_taxa_inadimplencia_por_grade(tabela_taxas: pd.DataFrame, coluna_categoria: str = "grade") -> plt.Figure:
    """Plota a taxa de inadimplência por grade com barras de erro (IC 95%).

    Parameters
    ----------
    tabela_taxas : pandas.DataFrame
        Saída de `calcular_taxa_inadimplencia_por_categoria`, contendo as
        colunas `taxa_inadimplencia`, `ic_inferior`, `ic_superior` e
        `n_contratos`.
    coluna_categoria : str, opcional
        Nome da coluna categórica que dá o eixo X do gráfico.

    Returns
    -------
    matplotlib.figure.Figure
        Gráfico de barras com a taxa de inadimplência por categoria e
        intervalo de confiança de 95%, além do volume de contratos por
        categoria em um eixo secundário. Uma relação monotônica
        crescente (A < B < C < ... < G) é o sinal esperado de um grade
        que discrimina bem o risco.
    """
    x = np.arange(len(tabela_taxas))
    y = tabela_taxas["taxa_inadimplencia"].values * 100
    erro_inferior = (tabela_taxas["taxa_inadimplencia"] - tabela_taxas["ic_inferior"]).values * 100
    erro_superior = (tabela_taxas["ic_superior"] - tabela_taxas["taxa_inadimplencia"]).values * 100

    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.bar(x, y, yerr=[erro_inferior, erro_superior], capsize=4, color="#C62828", alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(tabela_taxas[coluna_categoria].astype(str))
    ax1.set_ylabel("Taxa de inadimplência (%)")
    ax1.set_xlabel(coluna_categoria)
    ax1.set_title(f"Taxa de inadimplência por {coluna_categoria} (IC 95%)")

    ax2 = ax1.twinx()
    ax2.plot(x, tabela_taxas["n_contratos"], color="grey", marker="o", linestyle="--", alpha=0.6)
    ax2.set_ylabel("Nº de contratos", color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")

    fig.tight_layout()
    return fig


fig = plot_taxa_inadimplencia_por_grade(tabela_grade)
FIGURAS_PARA_PDF.append(fig)
plt.show()


In [ ]:
def testar_associacao_categorica(
    df: pd.DataFrame,
    coluna_categoria: str,
    coluna_alvo: str = CFG.coluna_alvo,
) -> dict:
    """Testa a associação entre uma coluna categórica e o alvo binário.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna_categoria : str
        Nome da coluna categórica a ser testada (ex.: `grade`).
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.

    Returns
    -------
    dict
        Dicionário com as chaves `qui_quadrado`, `p_valor`,
        `graus_liberdade` e `cramers_v`. O Cramér's V mede o tamanho do
        efeito da associação, em uma escala de 0 (nenhuma associação) a
        1 (associação perfeita), sendo mais informativo que o p-valor
        isolado quando o volume de dados é grande (nesse caso o
        p-valor tende a ficar artificialmente pequeno).
    """
    tabela_contingencia = pd.crosstab(df[coluna_categoria], df[coluna_alvo])
    chi2, p_valor, graus_liberdade, _ = stats.chi2_contingency(tabela_contingencia)

    n = tabela_contingencia.sum().sum()
    k = min(tabela_contingencia.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else np.nan

    return {
        "qui_quadrado": chi2,
        "p_valor": p_valor,
        "graus_liberdade": graus_liberdade,
        "cramers_v": cramers_v,
    }


def calcular_correlacao_ordinal(
    df: pd.DataFrame,
    coluna_categoria_ordinal: str,
    coluna_alvo: str = CFG.coluna_alvo,
    ordem: list = CFG.ordem_grades,
) -> dict:
    """Calcula a correlação de Spearman entre o rank de uma categoria ordinal e o alvo.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna_categoria_ordinal : str
        Nome da coluna categórica ordinal (ex.: `grade`).
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.
    ordem : list, opcional
        Ordem crescente de risco das categorias, usada para converter a
        categoria em um rank numérico antes de calcular a correlação.

    Returns
    -------
    dict
        Dicionário com `correlacao_spearman` e `p_valor`. Uma
        correlação positiva e significativa indica que a ordem do
        grade é, de fato, respeitada pela taxa de inadimplência
        observada (quanto pior o grade, maior o rank, maior a
        inadimplência).
    """
    mapa_rank = {categoria: rank for rank, categoria in enumerate(ordem, start=1)}
    rank_categoria = df[coluna_categoria_ordinal].map(mapa_rank)

    correlacao, p_valor = stats.spearmanr(rank_categoria, df[coluna_alvo])
    return {"correlacao_spearman": correlacao, "p_valor": p_valor}


resultado_associacao = testar_associacao_categorica(df, "grade")
resultado_correlacao = calcular_correlacao_ordinal(df, "grade")

print("Teste qui-quadrado de independência (grade x is_bad):")
chave_qui2 = resultado_associacao["qui_quadrado"]
chave_p1 = resultado_associacao["p_valor"]
chave_gl = resultado_associacao["graus_liberdade"]
chave_cramer = resultado_associacao["cramers_v"]
chave_spearman = resultado_correlacao["correlacao_spearman"]
chave_p2 = resultado_correlacao["p_valor"]

print("Teste qui-quadrado de independência (grade x is_bad):")
print(f"  qui-quadrado = {chave_qui2:.2f}")
print(f"  p-valor      = {chave_p1:.2e}")
print(f"  graus de liberdade = {chave_gl}")
print(f"  Cramér's V   = {chave_cramer:.3f}")
print()
print("Correlação de Spearman (rank do grade x is_bad):")
print(f"  correlação = {chave_spearman:.3f}")
print(f"  p-valor    = {chave_p2:.2e}")


**Como interpretar:**

O p-valor do qui-quadrado tende a ficar extremamente pequeno em bases grandes mesmo para
associações fracas, então ele serve só para confirmar que a associação não é fruto do acaso. O que
realmente importa aqui é o **Cramér's V** (tamanho do efeito) e a **correlação de Spearman** (que
captura diretamente se a ordem A → G é respeitada pela inadimplência observada). Um Spearman
positivo e alto é a evidência mais direta de que o grade cumpre bem o papel de score ordinal de
risco.


In [ ]:
def calcular_information_value(
    df: pd.DataFrame,
    coluna_categoria: str,
    coluna_alvo: str = CFG.coluna_alvo,
    ordem: list = None,
) -> pd.DataFrame:
    """Calcula o Weight of Evidence (WOE) e o Information Value (IV) por categoria.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna_categoria : str
        Nome da coluna categórica a ser avaliada como variável
        explicativa de risco.
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência (1 = evento, 0 = não
        evento).
    ordem : list, opcional
        Ordem de exibição das categorias no resultado.

    Returns
    -------
    pandas.DataFrame
        Uma linha por categoria, com contagem de bons, contagem de maus,
        WOE e contribuição para o IV total. O IV total (soma da coluna
        `iv_parcial`) é uma métrica padrão de crédito para avaliar poder
        preditivo de uma variável: valores acima de 0.3 costumam ser
        interpretados como poder preditivo forte, entre 0.1 e 0.3 como
        médio, e abaixo de 0.02 como praticamente nulo.

    Notes
    -----
    Para evitar divisão por zero em categorias sem nenhum mau pagador
    (ou sem nenhum bom pagador), aplicamos uma correção de Laplace
    somando 0.5 às contagens antes do cálculo do WOE.
    """
    tabela = df.groupby(coluna_categoria, observed=True)[coluna_alvo].agg(
        n_maus="sum", n_total="count"
    )
    tabela["n_bons"] = tabela["n_total"] - tabela["n_maus"]

    total_bons = tabela["n_bons"].sum()
    total_maus = tabela["n_maus"].sum()

    tabela["pct_bons"] = (tabela["n_bons"] + 0.5) / (total_bons + 0.5 * len(tabela))
    tabela["pct_maus"] = (tabela["n_maus"] + 0.5) / (total_maus + 0.5 * len(tabela))

    tabela["woe"] = np.log(tabela["pct_bons"] / tabela["pct_maus"])
    tabela["iv_parcial"] = (tabela["pct_bons"] - tabela["pct_maus"]) * tabela["woe"]

    if ordem is not None:
        tabela = tabela.reindex(ordem)

    return tabela.reset_index()


tabela_iv = calcular_information_value(df, "grade", ordem=CFG.ordem_grades)
iv_total = tabela_iv["iv_parcial"].sum()
print(f"Information Value total do grade: {iv_total:.3f}")
tabela_iv


In [ ]:
def plot_woe_por_categoria(tabela_iv: pd.DataFrame, coluna_categoria: str, iv_total: float) -> plt.Figure:
    """Plota o Weight of Evidence por categoria.

    Parameters
    ----------
    tabela_iv : pandas.DataFrame
        Saída de `calcular_information_value`, contendo a coluna `woe`.
    coluna_categoria : str
        Nome da coluna categórica que dá o eixo X do gráfico.
    iv_total : float
        Information Value total da variável, exibido no título do
        gráfico para dar contexto ao WOE por categoria.

    Returns
    -------
    matplotlib.figure.Figure
        Gráfico de barras do WOE por categoria. WOE positivo indica
        categoria com mais peso de "bom pagador" que a média da base;
        WOE negativo indica categoria com mais peso de "mau pagador".
        Uma progressão monotônica decrescente de A para G é o padrão
        esperado para uma variável de risco bem construída.
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    cores = [CFG.cor_bom if valor >= 0 else CFG.cor_mau for valor in tabela_iv["woe"]]
    ax.bar(tabela_iv[coluna_categoria].astype(str), tabela_iv["woe"], color=cores)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Weight of Evidence por {coluna_categoria} (IV total = {iv_total:.3f})")
    ax.set_ylabel("WOE")
    ax.set_xlabel(coluna_categoria)
    fig.tight_layout()
    return fig


fig = plot_woe_por_categoria(tabela_iv, "grade", iv_total)
FIGURAS_PARA_PDF.append(fig)
plt.show()


In [ ]:
def calcular_auc_gini_por_categoria_ordinal(
    df: pd.DataFrame,
    coluna_categoria_ordinal: str,
    coluna_alvo: str = CFG.coluna_alvo,
    ordem: list = CFG.ordem_grades,
) -> dict:
    """Calcula AUC e Gini tratando o rank da categoria ordinal como score de risco.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado.
    coluna_categoria_ordinal : str
        Nome da coluna categórica ordinal (ex.: `grade`).
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.
    ordem : list, opcional
        Ordem crescente de risco das categorias, convertida em rank
        numérico para servir de "score".

    Returns
    -------
    dict
        Dicionário com `auc`, `gini`, `taxa_falso_positivo` e
        `taxa_verdadeiro_positivo` (os dois últimos são arrays, para uso
        direto na curva ROC). O Gini é derivado da AUC pela relação
        Gini = 2 * AUC - 1, convenção usual em modelos de crédito.
    """
    mapa_rank = {categoria: rank for rank, categoria in enumerate(ordem, start=1)}
    score_risco = df[coluna_categoria_ordinal].map(mapa_rank)

    auc = roc_auc_score(df[coluna_alvo], score_risco)
    gini = 2 * auc - 1
    fpr, tpr, _ = roc_curve(df[coluna_alvo], score_risco)

    return {
        "auc": auc,
        "gini": gini,
        "taxa_falso_positivo": fpr,
        "taxa_verdadeiro_positivo": tpr,
    }


def plot_curva_roc(resultado_auc: dict, coluna_categoria_ordinal: str) -> plt.Figure:
    """Plota a curva ROC do grade tratado como score de risco.

    Parameters
    ----------
    resultado_auc : dict
        Saída de `calcular_auc_gini_por_categoria_ordinal`.
    coluna_categoria_ordinal : str
        Nome da coluna usada como score, exibido na legenda.

    Returns
    -------
    matplotlib.figure.Figure
        Curva ROC com a diagonal de referência (classificador
        aleatório) e a AUC/Gini no título. Quanto mais distante da
        diagonal, melhor o poder discriminatório do grade.
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(
        resultado_auc["taxa_falso_positivo"],
        resultado_auc["taxa_verdadeiro_positivo"],
        color="#1565C0", linewidth=2,
        label=f"{coluna_categoria_ordinal} (AUC = {resultado_auc['auc']:.3f})",
    )
    ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Classificador aleatório")
    ax.set_xlabel("Taxa de falso positivo")
    ax.set_ylabel("Taxa de verdadeiro positivo")
    titulo_roc = f"Curva ROC — {coluna_categoria_ordinal} como score de risco\n(Gini = {resultado_auc['gini']:.3f})"
    ax.set_title(titulo_roc)
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


resultado_auc_grade = calcular_auc_gini_por_categoria_ordinal(df, "grade")
print(f"AUC do grade:  {resultado_auc_grade['auc']:.3f}")
print(f"Gini do grade: {resultado_auc_grade['gini']:.3f}")

fig = plot_curva_roc(resultado_auc_grade, "grade")
FIGURAS_PARA_PDF.append(fig)
plt.show()


In [ ]:
def plot_taxa_inadimplencia_subgrade(df: pd.DataFrame, coluna_alvo: str = CFG.coluna_alvo) -> plt.Figure:
    """Plota a taxa de inadimplência por sub_grade, para checar granularidade fina.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame tratado, contendo a coluna `sub_grade`.
    coluna_alvo : str, opcional
        Nome da coluna binária de inadimplência.

    Returns
    -------
    matplotlib.figure.Figure
        Gráfico de barras com a taxa de inadimplência por sub_grade
        (A1, A2, ..., G5), ordenado alfabeticamente. Serve para
        verificar se a monotonicidade observada no grade "grosso" se
        mantém quando olhamos a granularidade mais fina que a
        LendingClub também disponibiliza.
    """
    ordem_subgrade = sorted(df["sub_grade"].dropna().unique())
    tabela = calcular_taxa_inadimplencia_por_categoria(df, "sub_grade", coluna_alvo, ordem=ordem_subgrade)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(tabela["sub_grade"], tabela["taxa_inadimplencia"] * 100, color="#6A1B9A")
    ax.set_title("Taxa de inadimplência por sub_grade")
    ax.set_ylabel("Taxa de inadimplência (%)")
    ax.set_xlabel("sub_grade")
    ax.tick_params(axis="x", rotation=90)
    fig.tight_layout()
    return fig


fig = plot_taxa_inadimplencia_subgrade(df)
FIGURAS_PARA_PDF.append(fig)
plt.show()


## 5. Conclusão da Pergunta 2

Juntando as quatro evidências (taxa de inadimplência por grade, Cramér's V, correlação de Spearman
e AUC/Gini), dá para formar uma conclusão consistente sobre a qualidade do `grade`:

- Se a taxa de inadimplência cresce de forma monotônica de A para G, com intervalos de confiança
  que não se sobrepõem entre grades distantes (ex.: A vs G), isso já é uma evidência visual forte
  de que o grade discrimina risco.
- O Cramér's V dá o tamanho do efeito da associação (não depende do volume de dados, ao contrário
  do p-valor), e a correlação de Spearman confirma que essa associação segue a ordem esperada.
- O Information Value (IV) é a métrica mais usada no mercado de crédito para julgar variáveis de
  risco: valores de IV acima de ~0.3 indicam variável com poder preditivo forte, e o padrão de WOE
  decrescente de A para G (bons pagadores concentrados nos grades melhores) é o comportamento
  esperado de uma variável de risco bem calibrada.
- A AUC/Gini, tratando o grade isoladamente como um score, mostra o quanto ele sozinho já separa
  bem inadimplentes de bons pagadores, mesmo sem nenhuma outra variável do dataset.
- O gráfico de sub_grade mostra se essa monotonicidade se mantém na granularidade mais fina, o que
  reforça (ou não) a robustez do critério de risco da LendingClub.

**Observação importante:** o próprio grade provavelmente foi construído usando algumas das
variáveis presentes nessa base (ex.: `int_rate`, que é diretamente definido a partir do grade, ou
variáveis de crédito como `dti`, `delinq_2yrs`, `fico_range`). Isso significa que avaliar o grade
comparando com `is_bad` é uma forma razoável de checar a qualidade do critério de risco da
LendingClub, mas não prova causalidade nem garante que o grade seja ótimo. Ele pode discriminar bem
e ainda assim deixar valor em cima da mesa (por exemplo, se outras variáveis explicativas
adicionarem poder preditivo incremental além do que o grade já captura, isso ficaria para uma etapa
posterior de modelagem).


## 6. Exportação dos gráficos para um único PDF

In [ ]:
def exportar_figuras_pdf(figuras: list, caminho_saida: Path) -> None:
    """Exporta uma lista de figuras matplotlib para um único arquivo PDF.

    Parameters
    ----------
    figuras : list of matplotlib.figure.Figure
        Figuras geradas ao longo do notebook, na ordem em que devem
        aparecer no PDF final.
    caminho_saida : Path
        Caminho do arquivo PDF de saída.

    Returns
    -------
    None
        A função tem efeito colateral de escrever o arquivo em disco;
        não retorna nenhum valor.
    """
    with PdfPages(caminho_saida) as pdf:
        for figura in figuras:
            pdf.savefig(figura)
    print(f"PDF salvo em: {caminho_saida.resolve()}")


exportar_figuras_pdf(FIGURAS_PARA_PDF, CFG.caminho_saida_pdf)
